# EMD Data Extraction Pipeline

This notebook prepares the `.npy` arrays consumed by `emd_classification_generic.ipynb`.

## What it does

| Step | Description |
|---|---|
| 1 | Load subject metadata and audio files |
| 2 | Filter subjects by class label (depression / healthy) |
| 3 | Load, convert to mono, and resample each recording |
| 4 | Concatenate all recordings within each class |
| 5 | **Remove silent regions longer than 0.5 s** from the concatenated signal |
| 6 | Segment the cleaned signal into non-overlapping 20 s windows |
| 7 | Apply **Empirical Mode Decomposition (EMD)** to each segment → 16 IMFs |
| 8 | Save the resulting arrays as `.npy` files |

## Output format

Two `.npy` files, one per class, each with shape:

```
(n_segments, N_IMFS, segment_samples)
  e.g. (42, 16, 200000)
```

These files are loaded directly by `emd_classification_generic.ipynb` via `DEP_DATA_PATH` / `HEALTH_DATA_PATH`.

## Required packages

```
pip install PyEMD librosa soundfile pandas numpy tqdm
```

> **DAIC-WOZ note:** Access requires a signed data-use agreement with the USC Institute for Creative Technologies (https://dcapswoz.ict.usc.edu/). Point `AUDIO_DIR` at the folder containing the `.wav` interview files and `METADATA_CSV` at the file with PHQ-8 scores.

## 1. Imports

Standard libraries plus:
- **`librosa`** — audio loading, resampling, and silence detection
- **`soundfile`** — fast `.wav` reading backend for librosa
- **`PyEMD`** — Empirical Mode Decomposition implementation
- **`tqdm`** — progress bars for long loops

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from PyEMD import EMD

print("Libraries loaded successfully.")
import PyEMD
print(f"  PyEMD   : {PyEMD.__version__}")
print(f"  librosa : {librosa.__version__}")
print(f"  numpy   : {np.__version__}")

## 2. Configuration

All user-defined parameters are set here. Adjust before running.

| Parameter | Description |
|---|---|
| `AUDIO_DIR` | Folder containing the raw `.wav` interview files |
| `METADATA_CSV` | CSV with at least `Participant_ID` and `PHQ_Score` columns |
| `OUTPUT_DIR` | Where the `.npy` files will be saved |
| `GROUP_LABEL` | Label used in filenames and prints (`'female'`, `'male'`, `'all'`) |
| `SEX_FILTER` | `'F'`, `'M'`, or `None` to disable sex filtering |
| `DEP_THRESHOLD` | PHQ-8 score ≥ this value → depression class (default: 10) |
| `TARGET_SR` | Sampling rate in Hz after resampling (default: 10 000) |
| `MAX_SILENCE_S` | Silence gaps **longer** than this (seconds) are removed after concatenation |
| `SILENCE_TOP_DB` | Energy threshold in dB below which a frame is considered silent |
| `SEGMENT_DURATION` | Duration of each analysis window in seconds (default: 20) |
| `N_IMFS` | Number of IMFs to retain per segment (default: 16) |
| `AUDIO_SUFFIX` | Suffix appended to participant ID to build the audio filename |
| `SEX_COLUMN` | Column name in metadata CSV for biological sex |
| `RANDOM_STATE` | Seed for reproducibility |

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────
AUDIO_DIR    = Path("path/to/audio_files")
METADATA_CSV = Path("path/to/metadata.csv")
OUTPUT_DIR   = Path("path/to/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GROUP LABEL ───────────────────────────────────────────────────────────────
GROUP_LABEL = "female"   # 'female', 'male', or 'all'

# ── METADATA COLUMN NAMES ────────────────────────────────────────────────────
ID_COLUMN  = "Participant_ID"
PHQ_COLUMN = "PHQ_Score"
SEX_COLUMN = "Gender"           # set to None to skip sex filtering

# ── AUDIO FILE NAMING ────────────────────────────────────────────────────────
AUDIO_SUFFIX = "_AUDIO.wav"

# ── CLASSIFICATION THRESHOLDS ────────────────────────────────────────────────
DEP_THRESHOLD = 10
SEX_FILTER    = 'F'     # 'F', 'M', or None

# ── SIGNAL PARAMETERS ────────────────────────────────────────────────────────
TARGET_SR        = 10_000
SEGMENT_DURATION = 20
SEGMENT_SAMPLES  = TARGET_SR * SEGMENT_DURATION   # 200 000

# ── SILENCE REMOVAL ──────────────────────────────────────────────────────────
MAX_SILENCE_S  = 0.5    # gaps longer than this (seconds) are removed
SILENCE_TOP_DB = 20     # frames this many dB below peak are considered silent

# ── EMD PARAMETERS ───────────────────────────────────────────────────────────
N_IMFS = 16

# ── MISC ─────────────────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── DERIVED OUTPUT PATHS ─────────────────────────────────────────────────────
OUT_DEP    = OUTPUT_DIR / f"{GROUP_LABEL}_dep_{SEGMENT_DURATION}_imfs.npy"
OUT_HEALTH = OUTPUT_DIR / f"{GROUP_LABEL}_health_{SEGMENT_DURATION}_imfs.npy"

print("Configuration summary")
print(f"  Group            : {GROUP_LABEL}  |  Sex filter: {SEX_FILTER}")
print(f"  PHQ threshold    : >= {DEP_THRESHOLD} → Depression")
print(f"  Target SR        : {TARGET_SR} Hz")
print(f"  Silence removal  : gaps > {MAX_SILENCE_S} s  (top_db={SILENCE_TOP_DB})")
print(f"  Segment          : {SEGMENT_DURATION} s = {SEGMENT_SAMPLES:,} samples")
print(f"  N_IMFS           : {N_IMFS}")
print(f"  Output (dep)     : {OUT_DEP}")
print(f"  Output (health)  : {OUT_HEALTH}")

## 3. Load and Filter Metadata

Read the participant metadata CSV and split subjects into two groups:
- **Depression** — PHQ score ≥ `DEP_THRESHOLD`
- **Healthy** — PHQ score < `DEP_THRESHOLD`

If `SEX_FILTER` is set, only subjects of that sex are retained. Subjects whose audio file is missing from `AUDIO_DIR` are excluded with a warning.

In [ ]:
metadata = pd.read_csv(METADATA_CSV)
print(f"Metadata loaded: {len(metadata)} subjects, columns: {list(metadata.columns)}")

if SEX_FILTER is not None and SEX_COLUMN in metadata.columns:
    metadata = metadata[metadata[SEX_COLUMN] == SEX_FILTER].copy()
    print(f"After sex filter ('{SEX_FILTER}'): {len(metadata)} subjects")

metadata['label'] = (metadata[PHQ_COLUMN] >= DEP_THRESHOLD).astype(int)
dep_meta    = metadata[metadata['label'] == 1].copy()
health_meta = metadata[metadata['label'] == 0].copy()

print(f"\nClass distribution:")
print(f"  Depression (PHQ >= {DEP_THRESHOLD}) : {len(dep_meta)} subjects")
print(f"  Healthy    (PHQ <  {DEP_THRESHOLD}) : {len(health_meta)} subjects")


def find_audio_file(subject_id, audio_dir, suffix):
    candidate = Path(audio_dir) / f"{subject_id}{suffix}"
    if candidate.exists():
        return candidate
    for ext in ('.wav', '.flac', '.mp3'):
        c2 = Path(audio_dir) / f"{subject_id}{ext}"
        if c2.exists():
            return c2
    return None


def filter_available(df):
    found, missing = [], []
    for _, row in df.iterrows():
        p = find_audio_file(row[ID_COLUMN], AUDIO_DIR, AUDIO_SUFFIX)
        if p:
            found.append((row[ID_COLUMN], p))
        else:
            missing.append(row[ID_COLUMN])
    if missing:
        print(f"  WARNING: {len(missing)} audio file(s) not found: {missing}")
    return found


print("\nChecking audio file availability...")
dep_files    = filter_available(dep_meta)
health_files = filter_available(health_meta)
print(f"  Available depression recordings : {len(dep_files)}")
print(f"  Available healthy   recordings  : {len(health_files)}")

## 4. Load and Resample Audio

Each audio file is:
1. Loaded as a mono waveform (channels averaged if stereo)
2. Resampled to `TARGET_SR` Hz using `librosa.resample`
3. Appended to the class-level signal list

A diagnostic plot shows the duration distribution of the loaded recordings.

In [ ]:
def load_recordings(file_list, target_sr=TARGET_SR, label=''):
    signals, durations_s = [], []
    for subj_id, path in tqdm(file_list, desc=f"Loading {label}"):
        try:
            y, sr = librosa.load(str(path), sr=None, mono=True)
            if sr != target_sr:
                y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
            signals.append(y.astype(np.float32))
            durations_s.append(len(y) / target_sr)
        except Exception as e:
            print(f"  ERROR loading {subj_id} ({path}): {e}")
    return signals, durations_s


dep_signals,    dep_durations    = load_recordings(dep_files,    label='Depression')
health_signals, health_durations = load_recordings(health_files, label='Healthy')

print(f"\nLoaded {len(dep_signals)} depression  recordings")
print(f"Loaded {len(health_signals)} healthy      recordings")

all_dur = dep_durations + health_durations
print(f"\nRecording duration (all subjects):")
print(f"  Min  : {min(all_dur)/60:.1f} min")
print(f"  Max  : {max(all_dur)/60:.1f} min")
print(f"  Mean : {np.mean(all_dur)/60:.1f} min")
print(f"  Total: {sum(all_dur)/60:.1f} min")

### 4.1 Duration Distribution Plot

Visualise the distribution of recording lengths per class. Shorter recordings produce fewer 20 s segments.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist([d/60 for d in dep_durations],    bins=20, alpha=0.7, label='Depression', color='#e57373')
ax.hist([d/60 for d in health_durations], bins=20, alpha=0.7, label='Healthy',    color='#64b5f6')
ax.axvline(SEGMENT_DURATION / 60, color='black', linestyle='--',
           linewidth=1.2, label=f'{SEGMENT_DURATION} s window')
ax.set_xlabel('Duration (minutes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Recording duration distribution — {GROUP_LABEL}', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 5. Concatenation

Following Contribution 2 of the thesis (Sánchez Corrales & Solé-Casals, 2025):

> *"The recordings of all subjects within the same group (depression or healthy, separated by sex) are concatenated into a single vector before segmentation into 20-second windows, so that individual identity is dissolved in the dataset."*

All per-subject signals within each class are joined end-to-end with `np.concatenate`, producing one long 1-D array per class. The result is passed to the silence removal step before segmentation.

In [ ]:
dep_concat    = np.concatenate(dep_signals).astype(np.float32)
health_concat = np.concatenate(health_signals).astype(np.float32)

print(f"Concatenated signal lengths:")
print(f"  Depression : {len(dep_concat):>12,} samples  ({len(dep_concat)/TARGET_SR/60:.1f} min)")
print(f"  Healthy    : {len(health_concat):>12,} samples  ({len(health_concat)/TARGET_SR/60:.1f} min)")

## 5.1 Silence Removal

Long silent pauses in the concatenated signal are uninformative for depression detection and, if left in place, can produce segments that consist mostly of silence — artificially inflating the number of usable 20 s windows while contributing no discriminative content.

### Strategy

Silence is detected using `librosa.effects.split`, which identifies **non-silent intervals** based on a decibel threshold (`SILENCE_TOP_DB`): any frame whose energy falls more than `SILENCE_TOP_DB` dB below the signal peak is labelled silent.

Only **gaps longer than `MAX_SILENCE_S` seconds (0.5 s)** are removed. Short pauses — natural hesitations, brief inter-word silences — are preserved, as they are part of the normal acoustic structure of speech and may carry prosodic information relevant to depression.

### Algorithm

1. Obtain non-silent intervals with `librosa.effects.split`
2. For each pair of consecutive non-silent intervals, compute the gap between them
3. If the gap ≤ `MAX_SILENCE_S` × `TARGET_SR` samples → keep the gap (short pause)
4. If the gap > `MAX_SILENCE_S` × `TARGET_SR` samples → discard the gap (long silence)
5. Reconstruct the signal by concatenating the retained chunks

> **Key parameter:** `SILENCE_TOP_DB = 20` means that frames at least 20 dB quieter than the loudest frame in the signal are treated as silent. Increase this value to be more aggressive (remove more), decrease it to be more conservative.

In [ ]:
def remove_long_silences(signal, sr, max_silence_s=MAX_SILENCE_S, top_db=SILENCE_TOP_DB):
    """
    Remove silent gaps longer than max_silence_s from a 1-D audio signal.
    Short pauses (duration <= max_silence_s) are preserved.

    Parameters
    ----------
    signal : np.ndarray, shape (n_samples,)
        Mono audio signal at sampling rate sr.
    sr : int
        Sampling rate in Hz.
    max_silence_s : float
        Maximum allowed silence gap in seconds. Gaps longer than this are removed.
    top_db : float
        Threshold (dB below peak) below which a frame is considered silent.
        Passed directly to librosa.effects.split.

    Returns
    -------
    cleaned : np.ndarray
        Signal with long silent gaps removed.
    n_removed_samples : int
        Total number of samples removed.
    n_gaps_removed : int
        Number of silent gaps that were removed.
    """
    max_gap_samples = int(max_silence_s * sr)

    # Non-silent intervals: array of [start, end] sample indices
    intervals = librosa.effects.split(signal, top_db=top_db)

    if len(intervals) == 0:
        # Entire signal is silent — return as-is with a warning
        print("  WARNING: no non-silent frames detected. Returning original signal.")
        return signal, 0, 0

    chunks          = []
    n_removed       = 0
    n_gaps_removed  = 0
    prev_end        = 0

    for start, end in intervals:
        gap = start - prev_end       # silence gap before this non-silent block

        if gap <= max_gap_samples:
            # Short pause: include the gap AND the non-silent block
            chunks.append(signal[prev_end:end])
        else:
            # Long silence: discard the gap, keep only the non-silent block
            chunks.append(signal[start:end])
            n_removed      += gap
            n_gaps_removed += 1

        prev_end = end

    # Trailing content after the last non-silent interval
    if prev_end < len(signal):
        trailing_gap = len(signal) - prev_end
        if trailing_gap <= max_gap_samples:
            chunks.append(signal[prev_end:])
        else:
            n_removed      += trailing_gap
            n_gaps_removed += 1

    cleaned = np.concatenate(chunks) if chunks else signal
    return cleaned, n_removed, n_gaps_removed


# ── Apply to both classes ────────────────────────────────────────────────────
print(f"Removing silence gaps > {MAX_SILENCE_S} s  (top_db={SILENCE_TOP_DB}) ...\n")

dep_clean, dep_n_rem, dep_gaps = remove_long_silences(
    dep_concat, TARGET_SR,
    max_silence_s=MAX_SILENCE_S, top_db=SILENCE_TOP_DB
)
health_clean, health_n_rem, health_gaps = remove_long_silences(
    health_concat, TARGET_SR,
    max_silence_s=MAX_SILENCE_S, top_db=SILENCE_TOP_DB
)

# ── Report ───────────────────────────────────────────────────────────────────
def _report(label, original, cleaned, n_rem, n_gaps):
    pct = 100 * n_rem / len(original) if len(original) > 0 else 0
    print(f"  {label}")
    print(f"    Before  : {len(original):>12,} samples  ({len(original)/TARGET_SR/60:.2f} min)")
    print(f"    Removed : {n_rem:>12,} samples  ({n_rem/TARGET_SR:.2f} s)  "
          f"across {n_gaps} gap(s)  [{pct:.1f}% of total]")
    print(f"    After   : {len(cleaned):>12,} samples  ({len(cleaned)/TARGET_SR/60:.2f} min)")
    print()

_report("Depression", dep_concat,    dep_clean,    dep_n_rem,    dep_gaps)
_report("Healthy",    health_concat,  health_clean, health_n_rem, health_gaps)

### 5.2 Silence Removal — Diagnostic Plot

Compare a 10-second excerpt of the original and cleaned concatenated signal side by side to confirm that long silent stretches have been removed while speech content is preserved.

In [ ]:
PLOT_SECONDS = 10
n_plot       = PLOT_SECONDS * TARGET_SR

fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharey='row')
t_orig  = np.linspace(0, PLOT_SECONDS, n_plot)
t_clean = np.linspace(0, PLOT_SECONDS, min(n_plot, len(dep_clean)))

# Depression row
axes[0, 0].plot(t_orig,  dep_concat[:n_plot],              color='#e57373', linewidth=0.4)
axes[0, 0].set_title("Depression — original (first 10 s)", fontsize=10)
axes[0, 1].plot(t_clean, dep_clean[:min(n_plot, len(dep_clean))], color='#e57373', linewidth=0.4)
axes[0, 1].set_title("Depression — after silence removal (first 10 s)", fontsize=10)

# Healthy row
t_hclean = np.linspace(0, PLOT_SECONDS, min(n_plot, len(health_clean)))
axes[1, 0].plot(t_orig,   health_concat[:n_plot],                    color='#64b5f6', linewidth=0.4)
axes[1, 0].set_title("Healthy — original (first 10 s)", fontsize=10)
axes[1, 1].plot(t_hclean, health_clean[:min(n_plot, len(health_clean))], color='#64b5f6', linewidth=0.4)
axes[1, 1].set_title("Healthy — after silence removal (first 10 s)", fontsize=10)

for ax in axes.flatten():
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.set_ylabel('Amplitude', fontsize=9)

fig.suptitle(f'Silence removal effect — {GROUP_LABEL}  '
             f'(gaps > {MAX_SILENCE_S} s removed, top_db={SILENCE_TOP_DB})',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6. Segmentation

The silence-cleaned concatenated signal is divided into non-overlapping windows of `SEGMENT_SAMPLES` = `TARGET_SR × SEGMENT_DURATION` samples (200 000 samples = 20 s at 10 kHz).

Any trailing samples that do not fill a complete window are discarded. The output is a 2-D array of shape `(n_segments, SEGMENT_SAMPLES)`.

In [ ]:
def segment_signal(signal, segment_samples=SEGMENT_SAMPLES):
    """
    Split a 1-D signal into non-overlapping segments of fixed length.
    Trailing samples that do not fill a complete segment are discarded.

    Returns
    -------
    segments : np.ndarray, shape (n_segments, segment_samples)
    """
    n_segs   = len(signal) // segment_samples
    usable   = n_segs * segment_samples
    segments = signal[:usable].reshape(n_segs, segment_samples)
    return segments


dep_segments    = segment_signal(dep_clean)
health_segments = segment_signal(health_clean)

print(f"Segmentation complete:")
print(f"  Depression : {dep_segments.shape}   "
      f"({dep_segments.shape[0]} segments × {SEGMENT_DURATION} s)")
print(f"  Healthy    : {health_segments.shape}   "
      f"({health_segments.shape[0]} segments × {SEGMENT_DURATION} s)")

### 6.1 Inspect a Random Segment

Plot one random 20 s waveform from each class to confirm the signal looks as expected after silence removal and segmentation.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
i_dep    = rng.integers(0, len(dep_segments))
i_health = rng.integers(0, len(health_segments))

t = np.linspace(0, SEGMENT_DURATION, SEGMENT_SAMPLES)

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].plot(t, dep_segments[i_dep],       color='#e57373', linewidth=0.5)
axes[0].set_title(f'Depression segment #{i_dep}', fontsize=11)
axes[1].plot(t, health_segments[i_health], color='#64b5f6', linewidth=0.5)
axes[1].set_title(f'Healthy segment #{i_health}', fontsize=11)
for ax in axes:
    ax.set_ylabel('Amplitude', fontsize=10)
axes[1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle(f'Sample 20 s segments after silence removal — {GROUP_LABEL}', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Empirical Mode Decomposition (EMD)

EMD decomposes each segment into **Intrinsic Mode Functions (IMFs)** — data-driven, adaptive oscillatory components ordered from highest to lowest frequency.

The `PyEMD` library is used. For each segment:
1. The `EMD` object decomposes the signal into as many IMFs as the algorithm finds
2. If fewer than `N_IMFS` are produced, the remaining rows are zero-padded
3. Only the first `N_IMFS` components are retained

**Output shape per class:** `(n_segments, N_IMFS, SEGMENT_SAMPLES)` — e.g. `(42, 16, 200000)`

> ⚠️ This is the most computationally intensive step. For large datasets, consider the optional parallel cell below.

In [ ]:
def apply_emd(segments, n_imfs=N_IMFS, label=''):
    """
    Apply EMD to each segment and return an array of shape
    (n_segments, n_imfs, segment_samples).
    """
    n_segs, seg_len = segments.shape
    imf_array = np.zeros((n_segs, n_imfs, seg_len), dtype=np.float32)
    emd = EMD()

    for i, seg in enumerate(tqdm(segments, desc=f"EMD [{label}]")):
        try:
            imfs = emd(seg.astype(np.float64))
            k    = min(len(imfs), n_imfs)
            imf_array[i, :k, :] = imfs[:k].astype(np.float32)
            if len(imfs) < n_imfs:
                print(f"  Segment {i}: only {len(imfs)} IMFs produced "
                      f"(rows {len(imfs)}–{n_imfs-1} zero-padded)")
        except Exception as e:
            print(f"  ERROR on segment {i}: {e} — segment left as zeros")

    return imf_array


print(f"Function 'apply_emd' defined. Will extract {N_IMFS} IMFs per segment.")

### 7.1 Run EMD — Depression Group

Apply EMD to all depression-class segments. Runtime depends heavily on segment length and number of segments; expect **several minutes to hours** for large datasets.

In [ ]:
print(f"Running EMD on {len(dep_segments)} depression segments ...")
dep_imfs = apply_emd(dep_segments, n_imfs=N_IMFS, label='Depression')
print(f"\nDepression IMF array shape: {dep_imfs.shape}")

### 7.2 Run EMD — Healthy Group

In [ ]:
print(f"Running EMD on {len(health_segments)} healthy segments ...")
health_imfs = apply_emd(health_segments, n_imfs=N_IMFS, label='Healthy')
print(f"\nHealthy IMF array shape: {health_imfs.shape}")

### 7.3 (Optional) Parallel EMD with `joblib`

For large datasets the sequential version above can be slow. The cell below uses `joblib.Parallel` to run EMD on multiple segments simultaneously. Uncomment and run instead of cells 7.1–7.2 if needed.

> **Note:** `PyEMD` is not thread-safe; use `backend='loky'` (multiprocessing) rather than `'threading'`.

In [ ]:
# ── OPTIONAL: parallel EMD ───────────────────────────────────────────────────
# from joblib import Parallel, delayed
#
# def _emd_one(seg, n_imfs):
#     emd  = EMD()
#     imfs = emd(seg.astype(np.float64))
#     out  = np.zeros((n_imfs, len(seg)), dtype=np.float32)
#     k    = min(len(imfs), n_imfs)
#     out[:k] = imfs[:k].astype(np.float32)
#     return out
#
# N_JOBS = -1
# dep_imfs = np.stack(Parallel(n_jobs=N_JOBS, backend='loky', verbose=5)(
#     delayed(_emd_one)(seg, N_IMFS) for seg in dep_segments))
# health_imfs = np.stack(Parallel(n_jobs=N_JOBS, backend='loky', verbose=5)(
#     delayed(_emd_one)(seg, N_IMFS) for seg in health_segments))
#
# print(f"Depression IMF array: {dep_imfs.shape}")
# print(f"Healthy IMF array   : {health_imfs.shape}")

print("Parallel cell defined (commented out by default).")

## 8. Inspect IMFs

Plot the 16 IMFs extracted from one random depression segment to confirm the decomposition looks reasonable. IMFs are ordered from highest frequency (IMF 1) to lowest (IMF 16 / residue).

In [ ]:
rng   = np.random.default_rng(RANDOM_STATE)
i_seg = rng.integers(0, len(dep_imfs))
t     = np.linspace(0, SEGMENT_DURATION, SEGMENT_SAMPLES)

fig, axes = plt.subplots(N_IMFS + 1, 1, figsize=(14, 2 * (N_IMFS + 1)), sharex=True)

axes[0].plot(t, dep_segments[i_seg], color='black', linewidth=0.4)
axes[0].set_ylabel('Signal', fontsize=9)
axes[0].set_title(f'Original signal + IMFs — depression segment #{i_seg}', fontsize=12)

for k in range(N_IMFS):
    axes[k + 1].plot(t, dep_imfs[i_seg, k], linewidth=0.4, color='#1976D2')
    axes[k + 1].set_ylabel(f'IMF {k+1}', fontsize=9)

axes[-1].set_xlabel('Time (s)', fontsize=10)
plt.tight_layout()
plt.show()

## 9. Save Arrays to Disk

Save both IMF arrays as NumPy `.npy` files. The output paths are constructed from the configuration variables so that the filenames reflect the group label and segment duration.

**These files are the direct inputs for `emd_classification_generic.ipynb`** — set `DEP_DATA_PATH` and `HEALTH_DATA_PATH` in that notebook to the paths printed below.

In [ ]:
np.save(OUT_DEP,    dep_imfs)
np.save(OUT_HEALTH, health_imfs)

dep_size    = OUT_DEP.stat().st_size    / 1e6
health_size = OUT_HEALTH.stat().st_size / 1e6

print("Arrays saved successfully.")
print(f"\n  Depression : {OUT_DEP}")
print(f"               shape={dep_imfs.shape}  |  {dep_size:.1f} MB")
print(f"\n  Healthy    : {OUT_HEALTH}")
print(f"               shape={health_imfs.shape}  |  {health_size:.1f} MB")
print()
print("Next step: paste these paths into emd_classification_generic.ipynb")
print(f"  DEP_DATA_PATH    = r\"{OUT_DEP}\"")
print(f"  HEALTH_DATA_PATH = r\"{OUT_HEALTH}\"")

## 10. Verification

Reload the saved files and confirm the shapes and value ranges are as expected before proceeding to the classification notebook.

In [ ]:
dep_check    = np.load(OUT_DEP,    allow_pickle=True)
health_check = np.load(OUT_HEALTH, allow_pickle=True)

print("Verification of saved files")
print(f"  {'File':<35} {'Shape':<25} {'dtype':<10} {'min':>10} {'max':>10}")
print(f"  {'-'*90}")
for name, arr in [('depression', dep_check), ('healthy', health_check)]:
    print(f"  {name:<35} {str(arr.shape):<25} {str(arr.dtype):<10} "
          f"{arr.min():>10.4f} {arr.max():>10.4f}")

assert dep_check.ndim    == 3, "Depression array must be 3-D"
assert health_check.ndim == 3, "Healthy array must be 3-D"
assert dep_check.shape[1]    == N_IMFS,         f"Expected {N_IMFS} IMFs"
assert health_check.shape[1] == N_IMFS,         f"Expected {N_IMFS} IMFs"
assert dep_check.shape[2]    == SEGMENT_SAMPLES, "Unexpected segment length (depression)"
assert health_check.shape[2] == SEGMENT_SAMPLES, "Unexpected segment length (healthy)"

print("\n✓ All shape assertions passed. Ready for classification notebook.")